# VERA · MimicGen **Stack** (client–server, minimalist)

Drives the MimicGen `stack_d0` two-block stacking task against a running VERA policy server.
Inference runs on the server's GPU; this notebook only steps the env.

**Start the server first** (stack prompt + VGGT IDM + cotracker), e.g.:
```bash
cd third_party/vera && export PYTHONPATH=$PWD
OPY=/path/to/miniconda3/envs/okto/bin/python
setsid env CUDA_VISIBLE_DEVICES=5 \
  VERA_TRACKER_BACKEND=cotracker VERA_MOTION_PLAN_SCALE=1.0 \
  VERA_DYNAMICS_PROJECT=jacobian-mimicgen VERA_DYNAMICS_RUN_ID=285ouq1q \
  $OPY -m vera.server.start_vera_server --embodiment mimicgen --port 8800 --vis-port 8801 \
  --text 'A robot arm stacks one block on top of another block' > /tmp/vera_server.log 2>&1 < /dev/null &
```
Watch the **live viewer at http://localhost:8801/** (dream · tracks · jacobian · per-chunk play bars) while rollouts run.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
from pathlib import Path

def _project_root(start=None):
    start = Path(start or '.').resolve()
    for p in [start, *start.parents]:
        if (p / '.git').exists():
            return p
    return start

ROOT = _project_root(Path.cwd())
VERA = ROOT / 'third_party' / 'vera'
for p in (VERA, ROOT / 'third_party' / 'mimicgen'):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
import numpy as np
print('vera path:', VERA)

In [ ]:
# --- config knobs ---
import os
HOST, PORT, VIS_PORT = '127.0.0.1', 8800, 8801
# Point at your downloaded MimicGen stack_d0 initial-states dataset (HF: amandlek/mimicgen_datasets).
# Override with the VERA_MIMICGEN_DATASET env var; default matches the SETUP.md download location.
DATASET = os.environ.get('VERA_MIMICGEN_DATASET',
                         '/workspace/VERA/vera-ckpts/mimicgen-data/core/stack_d0.hdf5')  # two-block stack
# Best initial condition from the performance sweep: demo_80 reached 90% SR (18/20) on stack_d0
# under the proven recipe. The policy starts at frame ~20 automatically (the context warmup below
# is context_frames-1 = 20). Set DEMO_KEYS = None to fall back to DEMO_INDICES / first NUM_DEMOS.
DEMO_KEYS    = ['demo_80']     # <- proven best starting state (90% in the sweep)
NUM_DEMOS    = 1
ROLLOUT_HORIZON = 700          # sweep used 700; stack_d0 needs the headroom to finish the stack
RENDER_SIZE  = 128
DEMO_INDICES = None            # used only when DEMO_KEYS is None; None = first NUM_DEMOS demos
PROMPT = 'A robot arm stacks one block on top of another block'  # per-rollout; change anytime, no server restart

In [ ]:
# --- Connect: attach to the running server + print what it's serving ---
from vera.server.protocol.websocket_policy_client import WebsocketClientPolicy
client = WebsocketClientPolicy(host=HOST, port=PORT)
meta = client.get_server_metadata()
view_keys = list(meta['view_keys'])
context_frames = int(meta.get('context_frames', 9))
print('planner (WAN):', meta.get('planner_model'))
print('IDM         :', meta.get('idm_model'))
print('views       :', view_keys, '| context_frames:', context_frames)
print('action_space:', meta.get('action_space'), '| H:', meta.get('action_horizon'), '| control_hz:', meta.get('control_hz'))
print(f'live viewer : http://localhost:{VIS_PORT}/')

In [ ]:
# --- Build the MimicGen runner + a RemotePolicy that forwards inference to the server ---
from vera.env_runner.mimicgen_runner import MimicgenRunner, MimicgenRunnerCfg
from vera.controller.run_mimicgen_eval import RemotePolicy
import torch

cfg = MimicgenRunnerCfg(
    env_name='mimicgen', dataset_path=DATASET, render_size=RENDER_SIZE,
    render_obs_key=view_keys, num_demos_to_run=NUM_DEMOS,
    max_episode_steps=ROLLOUT_HORIZON, n_repeat=1, action_scale=1.0,
    save_videos=True, save_trajectory=False, save_rrd=False,
    output_dir='outputs/vera_mimicgen_stack', use_stored_model=False,
    demo_warmup_steps=max(context_frames - 1, 0), log_step_debug=False,
)
runner = MimicgenRunner(cfg, device=torch.device('cpu'))
runner.setup_env()
remote = RemotePolicy(client, view_keys=view_keys,
                      view_widths=[RENDER_SIZE] * len(view_keys), context_frames=context_frames,
                      prompt=PROMPT)
print('runner + remote policy ready')

In [ ]:
# --- Run rollouts (watch http://localhost:8801/ live while this runs) ---
if DEMO_KEYS:
    options = {'demo_keys': list(DEMO_KEYS)}        # best IC from the sweep (demo_80, frame ~20)
elif DEMO_INDICES is not None:
    options = {'demo_indices': DEMO_INDICES}
else:
    options = None
results = runner.run(remote, run_tag='vera_stack', options=options)

succ    = np.asarray(results.get('env_successes', []), dtype=bool)
relaxed = np.asarray(results.get('relaxed_successes', succ), dtype=bool)
maxr    = np.asarray(results.get('max_rewards', []), dtype=float)
n = len(succ)
print(f'success:  {int(succ.sum())}/{n}' + (f' = {100*succ.mean():.0f}%' if n else ''))
print(f'relaxed:  {int(relaxed.sum())}/{n}')
print(f'maxR mean: {maxr.mean() if maxr.size else 0:.3f}   (sparse: >0 only on a completed stack)')
print('videos:', results.get('save_dir'))

In [ ]:
# --- Inline metrics + all videos (policy / agentview / wrist), like the reference notebook ---
from vera.env_runner.mimicgen_runner import format_run_results
r = format_run_results(results)
r.show(fps=5, height=252)
# Or access individually: r.metrics, r.videos['policy'], r.videos['agentview_image']

In [ ]:
# --- Composite policy-vis: [ current | dream + tracks -> dream | jacobian ] ---
# Snapshots the live viewer's policy_vis buffer (same panels as http://localhost:{VIS_PORT}/)
# from the rollout you just ran, and inlines it. Run RIGHT AFTER the rollout cell.
import urllib.request, json, time, base64, tempfile
import cv2, numpy as np, mediapy as media
from pathlib import Path
from IPython.display import HTML, display

def show_policy_vis(vis_host=HOST, vis_port=VIS_PORT, fps=10, max_width=1100):
    base = f"http://{vis_host}:{vis_port}"
    try:
        with urllib.request.urlopen(f"{base}/stats.json", timeout=5) as r:
            n = int(json.loads(r.read().decode()).get("video_len", 0))
    except Exception as e:
        print(f"[vis] server not reachable at {base} - is it up with --vis-port {vis_port}? ({e})"); return
    if n == 0:
        print("[vis] buffer empty - run the rollout cell first (watch the live viewer while it runs)."); return
    urllib.request.urlopen(f"{base}/video/pause", timeout=3)
    frames = []
    for i in range(n):
        urllib.request.urlopen(f"{base}/video/seek?pos={i}", timeout=3); time.sleep(0.05)
        with urllib.request.urlopen(f"{base}/policy.mjpg", timeout=5) as resp:
            buf = b""
            while True:
                chunk = resp.read(4096)
                if not chunk: break
                buf += chunk
                e = buf.find(b"\xff\xd9")
                if e != -1:
                    s = buf.find(b"\xff\xd8")
                    if s != -1:
                        img = cv2.imdecode(np.frombuffer(buf[s:e+2], np.uint8), cv2.IMREAD_COLOR)
                        if img is not None: frames.append(img)
                    break
    urllib.request.urlopen(f"{base}/video/play", timeout=3)
    if not frames:
        print("[vis] no frames grabbed."); return
    # Frames vary in HEIGHT (the plan-time first frame has an extra action panel at the bottom).
    # PAD shorter frames to the common size with black, top-aligned -- resizing would STRETCH and
    # warp the panels. Then encode H.264 via mediapy (cv2's mp4v fourcc renders a BLACK video).
    Hm = max(f.shape[0] for f in frames); Wm = max(f.shape[1] for f in frames)
    def _pad(f):
        h, w = f.shape[:2]
        if (h, w) == (Hm, Wm): return f
        out = np.zeros((Hm, Wm, 3), dtype=f.dtype); out[:h, :w] = f; return out
    rgb = np.stack([cv2.cvtColor(_pad(f), cv2.COLOR_BGR2RGB) for f in frames])
    mp4 = tempfile.mktemp(suffix=".mp4")
    media.write_video(mp4, rgb, fps=fps)
    b64 = base64.b64encode(Path(mp4).read_bytes()).decode()
    print(f"[vis] composite policy-vis: {len(rgb)} frames @ {Wm}x{Hm}")
    display(HTML(
        "<p><b>[ current | dream + tracks &#8594; dream | jacobian ]</b></p>"
        f'<video width="{min(Wm, max_width)}" controls loop autoplay muted>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

show_policy_vis()
